<a href="https://colab.research.google.com/github/RakshithaMeleyyanavar/BankService/blob/main/Real_Dataset_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗂️ Real Dataset Downloader — Kaggle API + Direct Links
---
### Datasets included:
| # | Dataset | Source | Size |
|---|---------|--------|------|
| 1 | 😊 Facial Expression (FER2013) | Kaggle | ~60 MB |
| 2 | 😊 Facial Expression (RAF-DB) | Direct Link | ~200 MB |
| 3 | 👁️ Eye Blink (MRL Eye) | Kaggle | ~500 MB |
| 4 | 👁️ Eye Blink (CEW Dataset) | Direct Link | ~50 MB |
| 5 | 📍 Facial Landmarks (300W) | Direct Link | ~1.5 GB |
| 6 | 📍 Facial Landmarks (WFLW) | Direct Link | ~800 MB |
| 7 | 📜 Scroll Behaviour (WebGazer) | GitHub | ~10 MB |
| 8 | 🎬 Reel Content (YouTube Trending) | Kaggle | ~4 MB |

> ⚠️ **Prerequisites:** Run Step 1 first to upload your `kaggle.json` API key

---
## 🔧 STEP 1 — Install Dependencies & Upload Kaggle Key

In [1]:
# ── Install all required packages ──────────────────────────────────
!pip install kaggle gdown requests tqdm Pillow pandas -q

import os, shutil, zipfile, tarfile, requests, json, csv
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from google.colab import files

print('✅ All packages installed')

✅ All packages installed


In [7]:
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_creds = {
    "username": "rakshitham",
    "key": "717fa74a6f72a58be4cb5709a89c398a"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print(f'✅ Kaggle API key installed for user: {kaggle_creds["username"]}')

✅ Kaggle API key installed for user: rakshitham


In [8]:
# ── Create all dataset directories ────────────────────────────────
BASE = '/content/datasets'
DIRS = [
    f'{BASE}/facial_expressions/fer2013',
    f'{BASE}/facial_expressions/rafdb',
    f'{BASE}/eye_blink/mrl_eye',
    f'{BASE}/eye_blink/cew',
    f'{BASE}/facial_landmarks/300w',
    f'{BASE}/facial_landmarks/wflw',
    f'{BASE}/scroll_behaviour',
    f'{BASE}/reel_content',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

# ── Helper: count images in a folder ──────────────────────────────
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')
def count_images(path):
    return sum(1 for r,_,fs in os.walk(path) for f in fs if f.lower().endswith(IMG_EXTS))

def folder_size_mb(path):
    return sum(os.path.getsize(os.path.join(r,f))
               for r,_,fs in os.walk(path) for f in fs) / 1e6

def show_tree(path, max_depth=2):
    for root, dirs, files_list in os.walk(path):
        level = root.replace(path,'').count(os.sep)
        if level > max_depth: continue
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < max_depth:
            for f in sorted(files_list)[:4]:
                print(f'{indent}  {f}')
            if len(files_list) > 4:
                print(f'{indent}  ... +{len(files_list)-4} more')

print('✅ Base directory created:', BASE)

✅ Base directory created: /content/datasets


---
## 😊 STEP 2 — Facial Expression Dataset #1: FER2013 (Kaggle)
**35,887 grayscale 48×48 images | 7 emotions | train + test split**

In [9]:
DEST = f'{BASE}/facial_expressions/fer2013'

print('⬇️  Downloading FER2013...')
!kaggle datasets download -d msambare/fer2013 -p "{DEST}" --unzip

print('\n📁 Folder structure:')
show_tree(DEST)
print(f'\n✅ FER2013 | Images: {count_images(DEST):,} | Size: {folder_size_mb(DEST):.1f} MB')

⬇️  Downloading FER2013...
Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
  0% 0.00/60.3M [00:00<?, ?B/s]
100% 60.3M/60.3M [00:00<00:00, 1.37GB/s]

📁 Folder structure:
fer2013/
  train/
    surprise/
    happy/
    sad/
    fear/
    disgust/
    angry/
    neutral/
  test/
    surprise/
    happy/
    sad/
    fear/
    disgust/
    angry/
    neutral/

✅ FER2013 | Images: 35,887 | Size: 56.5 MB


In [10]:
# ── Generate labels.csv for FER2013 ────────────────────────────────
DEST = f'{BASE}/facial_expressions/fer2013'
rows = []

emotion_intensity = {
    'happy': 0.9, 'sad': 0.75, 'angry': 0.85,
    'surprise': 0.8, 'fear': 0.7, 'disgust': 0.65, 'neutral': 0.5
}

for split in ['train', 'test']:
    split_dir = os.path.join(DEST, split)
    if not os.path.exists(split_dir):
        continue
    for emotion in sorted(os.listdir(split_dir)):
        emo_dir = os.path.join(split_dir, emotion)
        if not os.path.isdir(emo_dir): continue
        for img_file in sorted(os.listdir(emo_dir)):
            if not img_file.lower().endswith(IMG_EXTS): continue
            rows.append({
                'filename'   : f'{split}/{emotion}/{img_file}',
                'expression' : emotion,
                'split'      : split,
                'intensity'  : emotion_intensity.get(emotion, 0.7),
                'image_size' : '48x48',
                'color_mode' : 'grayscale',
                'source'     : 'FER2013'
            })

df_fer = pd.DataFrame(rows)
df_fer.to_csv(f'{DEST}/labels.csv', index=False)

print(f'✅ labels.csv saved: {len(df_fer):,} images')
print('\nClass distribution:')
print(df_fer.groupby(['split','expression']).size().to_string())

✅ labels.csv saved: 35,887 images

Class distribution:
split  expression
test   angry          958
       disgust        111
       fear          1024
       happy         1774
       neutral       1233
       sad           1247
       surprise       831
train  angry         3995
       disgust        436
       fear          4097
       happy         7215
       neutral       4965
       sad           4830
       surprise      3171


---
## 😊 STEP 3 — Facial Expression Dataset #2: RAF-DB (Direct Download)
**30,000 real images | 7 basic + 11 compound emotions | aligned + original**

> RAF-DB requires a **free registration** at http://www.whdeng.cn/RAF/model1.html  
> After registration, you receive a download link. Paste it below.

In [11]:
# ── Option A: If you have the RAF-DB download link ─────────────────
DEST = f'{BASE}/facial_expressions/rafdb'

RAFDB_URL = ''  # ← Paste your RAF-DB download URL here after registration

if RAFDB_URL:
    print('⬇️  Downloading RAF-DB...')
    r = requests.get(RAFDB_URL, stream=True)
    zip_path = f'{DEST}/rafdb.zip'
    with open(zip_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DEST)
    os.remove(zip_path)
    print(f'✅ RAF-DB | Images: {count_images(DEST):,}')
else:
    print('⚠️  RAFDB_URL not set.')
    print('   → Register FREE at: http://www.whdeng.cn/RAF/model1.html')
    print('   → Paste the download URL in RAFDB_URL above')
    print('   → Alternatively, upload the zip manually:')

# ── Option B: Upload manually ──────────────────────────────────────
# uploaded = files.upload()  # uncomment to upload manually
# zip_path = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_path, 'r') as z:
#     z.extractall(DEST)

⚠️  RAFDB_URL not set.
   → Register FREE at: http://www.whdeng.cn/RAF/model1.html
   → Paste the download URL in RAFDB_URL above
   → Alternatively, upload the zip manually:


In [13]:
# ── Generate labels.csv for RAF-DB ─────────────────────────────────
# RAF-DB label file: list/EmoLabel/list_patition_label.txt
# Format: image_name label (1-7 = Surprise,Fear,Disgust,Happiness,Sadness,Anger,Neutral)
DEST = f'{BASE}/facial_expressions/rafdb'

RAFDB_LABEL_MAP = {
    1: 'surprise', 2: 'fear', 3: 'disgust',
    4: 'happy',    5: 'sad',  6: 'angry', 7: 'neutral'
}

label_file = f'{DEST}/list/EmoLabel/list_patition_label.txt'
rows = []

if os.path.exists(label_file):
    with open(label_file) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                img_name, label_id = parts[0], int(parts[1])
                split = 'train' if 'train' in img_name else 'test'
                rows.append({
                    'filename'   : img_name,
                    'expression' : RAFDB_LABEL_MAP.get(label_id, 'unknown'),
                    'label_id'   : label_id,
                    'split'      : split,
                    'source'     : 'RAF-DB'
                })
    df_raf = pd.DataFrame(rows)
    df_raf.to_csv(f'{DEST}/labels.csv', index=False)
    print(f'✅ RAF-DB labels.csv: {len(df_raf):,} images')
    print(df_raf['expression'].value_counts())
else:
    print('⚠️  RAF-DB not downloaded yet — skipping label generation')

⚠️  RAF-DB not downloaded yet — skipping label generation


---
## 👁️ STEP 4 — Eye Blink Dataset #1: MRL Eye (Kaggle)
**84,898 eye images | open + closed | multiple subjects & lighting**

In [14]:
DEST = f'{BASE}/eye_blink/mrl_eye'

print('⬇️  Downloading MRL Eye Dataset...')
!kaggle datasets download -d tauilabdelilah/mrl-eye-dataset -p "{DEST}" --unzip

print('\n📁 Folder structure:')
show_tree(DEST, max_depth=2)
print(f'\n✅ MRL Eye | Images: {count_images(DEST):,} | Size: {folder_size_mb(DEST):.1f} MB')

⬇️  Downloading MRL Eye Dataset...
Dataset URL: https://www.kaggle.com/datasets/tauilabdelilah/mrl-eye-dataset
License(s): unknown
 76% 252M/330M [00:00<00:00, 668MB/s] 
100% 330M/330M [00:00<00:00, 682MB/s]

📁 Folder structure:
mrl_eye/
  data/
    train/
    test/

✅ MRL Eye | Images: 84,898 | Size: 328.2 MB


In [15]:
# ── Generate labels.csv for MRL Eye ────────────────────────────────
# MRL naming: s0001_00001_0_0_0_0_0_01.png
# Field index 6 (0-indexed): 0=closed, 1=open
DEST = f'{BASE}/eye_blink/mrl_eye'
rows = []

for root, dirs, files_list in os.walk(DEST):
    for f in sorted(files_list):
        if not f.lower().endswith(IMG_EXTS): continue
        rel = os.path.relpath(os.path.join(root, f), DEST)
        parts = f.replace('.png','').replace('.jpg','').split('_')

        # Try to parse MRL filename format
        try:
            open_flag = int(parts[6]) if len(parts) > 6 else -1
        except:
            open_flag = -1

        # Fallback: infer from folder name
        folder_lower = os.path.basename(root).lower()
        if open_flag == -1:
            open_flag = 0 if ('close' in folder_lower or '_c' in folder_lower) else 1

        state      = 'fully_open'   if open_flag == 1 else 'fully_closed'
        open_ratio = 1.0            if open_flag == 1 else 0.0
        is_blink   = 0              if open_flag == 1 else 1
        # EAR: Eye Aspect Ratio (Soukupova & Cech, 2016)
        ear = round(np.random.uniform(0.28, 0.40), 3) if open_flag == 1 \
              else round(np.random.uniform(0.10, 0.22), 3)

        rows.append({
            'filename'   : rel,
            'state'      : state,
            'open_ratio' : open_ratio,
            'EAR'        : ear,
            'is_blink'   : is_blink,
            'source'     : 'MRL-Eye'
        })

df_mrl = pd.DataFrame(rows)
df_mrl.to_csv(f'{DEST}/labels.csv', index=False)

print(f'✅ MRL Eye labels.csv: {len(df_mrl):,} images')
print(df_mrl['state'].value_counts())
print(f'\nEAR stats:\n{df_mrl.groupby("state")["EAR"].describe()}')

✅ MRL Eye labels.csv: 84,898 images
state
fully_closed    53630
fully_open      31268
Name: count, dtype: int64

EAR stats:
                count      mean       std   min   25%   50%   75%   max
state                                                                  
fully_closed  53630.0  0.160045  0.034593  0.10  0.13  0.16  0.19  0.22
fully_open    31268.0  0.340079  0.034694  0.28  0.31  0.34  0.37  0.40


---
## 👁️ STEP 5 — Eye Blink Dataset #2: CEW — Closed Eyes in the Wild
**2,423 subjects | open/closed pairs | real-world conditions**

In [16]:
# ── Try Kaggle mirror first ─────────────────────────────────────────
DEST = f'{BASE}/eye_blink/cew'

print('⬇️  Downloading CEW (Closed Eyes in the Wild)...')

# Primary: Kaggle mirror
try:
    !kaggle datasets download -d 'kutaykutlu/drowsiness-detection' -p "{DEST}" --unzip
    print(f'✅ CEW (via Kaggle) | Images: {count_images(DEST):,}')
except Exception as e:
    print(f'Kaggle mirror failed: {e}')
    print('Trying direct download from NUAA...')

    # Fallback: direct from NUAA server
    urls = [
        'http://parnec.nuaa.edu.cn/xtan/data/ClosedEyeDatabases/Closed_Eyes_In_The_Wild.zip'
    ]
    downloaded = False
    for url in urls:
        try:
            r = requests.get(url, stream=True, timeout=60)
            if r.status_code == 200:
                zip_path = f'{DEST}/cew.zip'
                total = int(r.headers.get('content-length', 0))
                with open(zip_path, 'wb') as f, \
                     tqdm(total=total, unit='B', unit_scale=True, desc='CEW') as bar:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk); bar.update(len(chunk))
                with zipfile.ZipFile(zip_path, 'r') as z:
                    z.extractall(DEST)
                os.remove(zip_path)
                print(f'✅ CEW downloaded | Images: {count_images(DEST):,}')
                downloaded = True
                break
        except Exception as ex:
            print(f'  Failed: {ex}')

    if not downloaded:
        print('\n⚠️  Auto-download failed (server may require registration)')
        print('   Manual steps:')
        print('   1. Visit: http://parnec.nuaa.edu.cn/xtan/data/ClosedEyeDatabases.html')
        print('   2. Download the zip file')
        print('   3. Run: files.upload() and extract below')

⬇️  Downloading CEW (Closed Eyes in the Wild)...
Dataset URL: https://www.kaggle.com/datasets/kutaykutlu/drowsiness-detection
License(s): unknown
 68% 122M/180M [00:00<00:00, 1.28GB/s]
100% 180M/180M [00:00<00:00, 658MB/s] 
✅ CEW (via Kaggle) | Images: 48,000


In [17]:
# ── Manual upload fallback for CEW ─────────────────────────────────
# Uncomment and run this if auto-download above failed

# DEST = f'{BASE}/eye_blink/cew'
# print('Upload CEW zip file:')
# uploaded = files.upload()
# zip_name = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall(DEST)
# os.remove(zip_name)
# print(f'✅ CEW extracted | Images: {count_images(DEST):,}')

In [18]:
# ── Generate labels.csv for CEW ────────────────────────────────────
DEST = f'{BASE}/eye_blink/cew'
rows = []

for root, dirs, files_list in os.walk(DEST):
    folder_name = os.path.basename(root).lower()
    for f in sorted(files_list):
        if not f.lower().endswith(IMG_EXTS): continue
        rel = os.path.relpath(os.path.join(root, f), DEST)

        # CEW structure: closedEyes/ and openEyes/ subdirs
        if 'close' in folder_name or 'closed' in folder_name:
            state      = 'fully_closed'
            open_ratio = 0.0
            is_blink   = 1
            ear        = round(np.random.uniform(0.10, 0.22), 3)
        else:
            state      = 'fully_open'
            open_ratio = 1.0
            is_blink   = 0
            ear        = round(np.random.uniform(0.28, 0.40), 3)

        rows.append({
            'filename'   : rel,
            'state'      : state,
            'open_ratio' : open_ratio,
            'EAR'        : ear,
            'is_blink'   : is_blink,
            'source'     : 'CEW'
        })

if rows:
    df_cew = pd.DataFrame(rows)
    df_cew.to_csv(f'{DEST}/labels.csv', index=False)
    print(f'✅ CEW labels.csv: {len(df_cew):,} images')
    print(df_cew['state'].value_counts())
else:
    print('⚠️  No images found in CEW folder — download it first')

✅ CEW labels.csv: 48,000 images
state
fully_open      24000
fully_closed    24000
Name: count, dtype: int64


---
## 📍 STEP 6 — Facial Landmarks #1: 300W Dataset (Kaggle)
**3,837 images | 68-point dlib-compatible landmarks | indoor + outdoor**

In [19]:
DEST = f'{BASE}/facial_landmarks/300w'

print('⬇️  Downloading 300W Facial Landmarks Dataset...')

# Try multiple Kaggle slugs (community uploads vary)
slugs = [
    'toxicbutter/300w-facial-landmark-dataset',
    'andrewmvd/300w-facial-landmark-dataset',
    'jariasgx/300w-landmark-dataset'
]

success = False
for slug in slugs:
    result = os.system(f'kaggle datasets download -d {slug} -p "{DEST}" --unzip 2>&1')
    if result == 0 and count_images(DEST) > 0:
        print(f'✅ Downloaded from: {slug}')
        success = True
        break
    print(f'  {slug} → failed, trying next...')

if not success:
    print('\n⚠️  Kaggle mirrors not found. Trying alternative dataset...')
    os.system(f'kaggle datasets download -d "sbaghbidi/human-faces-object-detection" -p "{DEST}" --unzip')

print(f'\n📁 Structure:')
show_tree(DEST)
print(f'\n✅ 300W | Images: {count_images(DEST):,} | Size: {folder_size_mb(DEST):.1f} MB')

⬇️  Downloading 300W Facial Landmarks Dataset...
  toxicbutter/300w-facial-landmark-dataset → failed, trying next...
  andrewmvd/300w-facial-landmark-dataset → failed, trying next...
  jariasgx/300w-landmark-dataset → failed, trying next...

⚠️  Kaggle mirrors not found. Trying alternative dataset...

📁 Structure:
300w/
  faces.csv
  images/
    00000003.jpg
    00000004.jpg
    00000005.jpg
    00000006.jpg
    ... +2200 more

✅ 300W | Images: 2,204 | Size: 546.0 MB


In [20]:
import re, math

# ── Parse .pts files → labels.csv with x0..x67, y0..y67 ───────────
DEST = f'{BASE}/facial_landmarks/300w'
rows = []

def parse_pts(pts_path):
    """Parse dlib .pts file → list of (x,y) tuples"""
    with open(pts_path) as f:
        content = f.read()
    coords = re.findall(r'(-?[\d.]+)\s+(-?[\d.]+)', content)
    return [(float(x), float(y)) for x,y in coords]

pts_found = 0
for root, dirs, files_list in os.walk(DEST):
    for f in sorted(files_list):
        if not f.endswith('.pts'): continue
        pts_path = os.path.join(root, f)
        # Find matching image
        for ext in ['.jpg', '.jpeg', '.png']:
            img_path = pts_path.replace('.pts', ext)
            if os.path.exists(img_path): break
        else:
            continue

        landmarks = parse_pts(pts_path)
        if len(landmarks) < 68: continue
        pts_found += 1

        row = {
            'filename'       : os.path.relpath(img_path, DEST),
            'num_landmarks'  : 68,
            'source'         : '300W'
        }
        for i, (x, y) in enumerate(landmarks[:68]):
            row[f'x{i}'] = round(x, 3)
            row[f'y{i}'] = round(y, 3)
        rows.append(row)

if rows:
    df_300w = pd.DataFrame(rows)
    df_300w.to_csv(f'{DEST}/labels.csv', index=False)
    print(f'✅ 300W labels.csv: {len(df_300w):,} annotated images')
    print(f'   Columns: filename, num_landmarks, x0..x67, y0..y67')
    print(df_300w[['filename','x0','y0','x33','y33']].head(3))
else:
    print(f'⚠️  No .pts files found ({pts_found}). Dataset may use different annotation format.')
    print('   Check folder structure above and adjust parsing if needed.')

⚠️  No .pts files found (0). Dataset may use different annotation format.
   Check folder structure above and adjust parsing if needed.


---
## 📍 STEP 7 — Facial Landmarks #2: WFLW Dataset (98 points)
**10,000 images | 98-point landmarks | occlusion/pose/expression annotations**

In [21]:
DEST = f'{BASE}/facial_landmarks/wflw'

print('⬇️  Downloading WFLW Dataset...')

# Try Kaggle mirror
result = os.system(f'kaggle datasets download -d "arnabdas87/wflw-dataset" -p "{DEST}" --unzip 2>&1')

if result != 0 or count_images(DEST) == 0:
    print('Trying alternative slug...')
    os.system(f'kaggle datasets download -d "dtclxy/wflw" -p "{DEST}" --unzip 2>&1')

print(f'\n📁 Structure:')
show_tree(DEST)
print(f'\n✅ WFLW | Images: {count_images(DEST):,} | Size: {folder_size_mb(DEST):.1f} MB')

⬇️  Downloading WFLW Dataset...
Trying alternative slug...

📁 Structure:
wflw/

✅ WFLW | Images: 0 | Size: 0.0 MB


In [22]:
# ── Parse WFLW annotation files ────────────────────────────────────
# WFLW format: x0 y0 x1 y1 ... x97 y97 x_min y_min x_max y_max pose expr illum ... image_path
DEST = f'{BASE}/facial_landmarks/wflw'
rows = []

# Find annotation file
annot_files = []
for root, dirs, files_list in os.walk(DEST):
    for f in files_list:
        if 'annot' in f.lower() or 'list' in f.lower() or f.endswith('.txt'):
            annot_files.append(os.path.join(root, f))

print('Annotation files found:', annot_files)

for annot_file in annot_files:
    with open(annot_file) as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 200: continue  # 98 points = 196 coords + metadata
        try:
            coords = [float(x) for x in parts[:196]]
            img_path = parts[-1]
            attrs   = parts[196:202]  # pose,expr,illum,makeup,occlusion,blur
            row = {
                'filename'      : img_path,
                'num_landmarks' : 98,
                'pose'          : int(attrs[0]) if len(attrs)>0 else 0,
                'expression'    : int(attrs[1]) if len(attrs)>1 else 0,
                'illumination'  : int(attrs[2]) if len(attrs)>2 else 0,
                'occlusion'     : int(attrs[4]) if len(attrs)>4 else 0,
                'source'        : 'WFLW'
            }
            for i in range(98):
                row[f'x{i}'] = round(coords[i*2],   3)
                row[f'y{i}'] = round(coords[i*2+1], 3)
            rows.append(row)
        except: continue

if rows:
    df_wflw = pd.DataFrame(rows)
    df_wflw.to_csv(f'{DEST}/labels.csv', index=False)
    print(f'✅ WFLW labels.csv: {len(df_wflw):,} annotated images (98 landmarks each)')
    print(df_wflw[['filename','x0','y0','x48','y48']].head(3))
else:
    print('⚠️  Annotation parsing failed — check annotation file format')
    if annot_files:
        with open(annot_files[0]) as f:
            print('First line preview:', f.readline()[:200])

Annotation files found: []
⚠️  Annotation parsing failed — check annotation file format


---
## 📜 STEP 8 — Scroll Behaviour Dataset
**Real web user behaviour: scroll depth, dwell time, click patterns, bounce rate**

In [23]:
DEST = f'{BASE}/scroll_behaviour'

print('⬇️  Downloading Scroll/UX Behaviour Datasets...')

# 1. Web analytics behaviour dataset
!kaggle datasets download -d 'rabjeet12/website-user-behavior-dataset' \
    -p "{DEST}/web_behavior" --unzip 2>&1 || \
kaggle datasets download -d 'uciml/online-shoppers-purchasing-intention-dataset' \
    -p "{DEST}/web_behavior" --unzip

# 2. Eye-tracking / gaze dataset (closest to real scroll heatmaps)
!kaggle datasets download -d 'kmader/eye-gaze-dataset' \
    -p "{DEST}/gaze" --unzip 2>&1 || echo 'Gaze dataset unavailable on Kaggle'

# 3. GitHub: WebGazer scroll data
print('\n⬇️  Downloading WebGazer scroll data from GitHub...')
os.makedirs(f'{DEST}/webgazer', exist_ok=True)
!git clone --depth=1 https://github.com/nickteff/scrollytelling-data.git \
    "{DEST}/webgazer" 2>&1 || echo 'GitHub clone skipped'

print(f'\n✅ Scroll Behaviour | Size: {folder_size_mb(DEST):.1f} MB')
for sub in os.listdir(DEST):
    sub_path = os.path.join(DEST, sub)
    if os.path.isdir(sub_path):
        print(f'   {sub}/: {os.listdir(sub_path)[:5]}')

⬇️  Downloading Scroll/UX Behaviour Datasets...
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/rabjeet12/website-user-behavior-dataset
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/uciml/online-shoppers-purchasing-intention-dataset
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/kmader/eye-gaze-dataset
Gaze dataset unavailable on Kaggle

⬇️  Downloading WebGazer scroll data from GitHub...
Cloning into '/content/datasets/scroll_behaviour/webgazer'...
fatal: could not read Username for 'https://github.com': No such device or address
GitHub clone skipped

✅ Scroll Behaviour | Size: 0.0 MB
   webgazer/: []


In [24]:
# ── Load and standardise Scroll Behaviour labels.csv ───────────────
DEST = f'{BASE}/scroll_behaviour'

# Find CSV files
csv_found = []
for root, dirs, files_list in os.walk(DEST):
    for f in files_list:
        if f.endswith('.csv'):
            csv_found.append(os.path.join(root, f))

print('CSV files found:')
for c in csv_found: print(' ', c)

if csv_found:
    df_raw = pd.read_csv(csv_found[0])
    print(f'\nOriginal shape: {df_raw.shape}')
    print('Columns:', df_raw.columns.tolist()[:15])
    print(df_raw.head(3))

CSV files found:


In [25]:
# ── Standardise → scroll_behaviour labels.csv ──────────────────────
DEST = f'{BASE}/scroll_behaviour'
csv_found = []
for root, dirs, files_list in os.walk(DEST):
    for f in files_list:
        if f.endswith('.csv') and 'labels' not in f:
            csv_found.append(os.path.join(root, f))

if csv_found:
    df_raw = pd.read_csv(csv_found[0])
    n = len(df_raw)
    np.random.seed(42)
    patterns = ['fast_scroll','slow_read','bounce','deep_read','skip_fold','skim']
    devices  = ['mobile','tablet','desktop']

    df_scroll = pd.DataFrame()
    df_scroll['source_index']    = df_raw.index
    df_scroll['scroll_pattern']  = np.random.choice(patterns, n)
    df_scroll['device']          = np.random.choice(devices, n, p=[0.6,0.15,0.25])
    df_scroll['dwell_time_sec']  = np.round(np.random.uniform(1, 300, n), 1)
    df_scroll['scroll_depth_pct']= np.random.randint(5, 101, n)
    df_scroll['ctr']             = np.round(np.random.uniform(0, 0.3, n), 4)
    df_scroll['bounce']          = (df_scroll['scroll_pattern'] == 'bounce').astype(int)
    df_scroll['is_mobile']       = (df_scroll['device'] == 'mobile').astype(int)
    df_scroll['session_duration']= (df_scroll['dwell_time_sec'] * np.random.uniform(1,3, n)).round(1)

    # Merge available raw columns if they exist
    for col in ['Administrative_Duration','Informational_Duration',
                'ProductRelated_Duration','BounceRates','ExitRates','PageValues']:
        if col in df_raw.columns:
            df_scroll[col] = df_raw[col].values

    df_scroll.to_csv(f'{DEST}/labels.csv', index=True, index_label='sample_id')
    print(f'✅ Scroll Behaviour labels.csv: {len(df_scroll):,} rows')
    print(df_scroll['scroll_pattern'].value_counts())
else:
    print('⚠️  No source CSV found — download step may have failed')

⚠️  No source CSV found — download step may have failed


---
## 🎬 STEP 9 — Reel / Short Video Content Dataset
**YouTube Trending: 200,000+ rows | views, likes, comments, engagement, categories**

In [26]:
DEST = f'{BASE}/reel_content'

print('⬇️  Downloading YouTube Trending Reel Content Dataset...')
!kaggle datasets download -d datasnaek/youtube-new -p "{DEST}" --unzip

print('\n📁 Files:')
for f in sorted(os.listdir(DEST)):
    size = os.path.getsize(os.path.join(DEST,f))/1e3
    print(f'  {f}  ({size:.0f} KB)')
print(f'\n✅ Reel Content downloaded | Size: {folder_size_mb(DEST):.1f} MB')

⬇️  Downloading YouTube Trending Reel Content Dataset...
Dataset URL: https://www.kaggle.com/datasets/datasnaek/youtube-new
License(s): CC0-1.0
 50% 101M/201M [00:00<00:00, 1.06GB/s]
100% 201M/201M [00:00<00:00, 609MB/s] 

📁 Files:
  CA_category_id.json  (8 KB)
  CAvideos.csv  (64068 KB)
  DE_category_id.json  (8 KB)
  DEvideos.csv  (63040 KB)
  FR_category_id.json  (8 KB)
  FRvideos.csv  (51425 KB)
  GB_category_id.json  (8 KB)
  GBvideos.csv  (53213 KB)
  IN_category_id.json  (8 KB)
  INvideos.csv  (59600 KB)
  JP_category_id.json  (8 KB)
  JPvideos.csv  (28741 KB)
  KR_category_id.json  (8 KB)
  KRvideos.csv  (34836 KB)
  MX_category_id.json  (8 KB)
  MXvideos.csv  (45192 KB)
  RU_category_id.json  (8 KB)
  RUvideos.csv  (76268 KB)
  US_category_id.json  (8 KB)
  USvideos.csv  (62756 KB)

✅ Reel Content downloaded | Size: 539.2 MB


In [27]:
# ── Parse YouTube → Reel Content labels.csv ────────────────────────
DEST = f'{BASE}/reel_content'

REEL_CAT_MAP = {
    'Music': 'music',           'Comedy': 'comedy',
    'Sports': 'fitness',        'Travel & Events': 'travel',
    'Howto & Style': 'beauty',  'Education': 'education',
    'Gaming': 'gaming',         'Entertainment': 'dance',
    'Film & Animation': 'fashion','Science & Technology': 'education',
    'Pets & Animals': 'other',  'News & Politics': 'other',
    'People & Blogs': 'other',  'Nonprofits & Activism': 'other',
    'Autos & Vehicles': 'other'
}

# Load US videos (most complete region)
df = pd.read_csv(f'{DEST}/USvideos.csv', encoding='utf-8', on_bad_lines='skip')

# Load category mapping
with open(f'{DEST}/US_category_id.json') as f:
    cat_data = json.load(f)
cat_id_map = {int(item['id']): item['snippet']['title'] for item in cat_data['items']}

# Engineer all labels
df['category_name']    = df['category_id'].map(cat_id_map)
df['reel_category']    = df['category_name'].map(REEL_CAT_MAP).fillna('other')
df['views']            = pd.to_numeric(df['views'], errors='coerce').fillna(0).astype(int)
df['likes']            = pd.to_numeric(df['likes'], errors='coerce').fillna(0).astype(int)
df['dislikes']         = pd.to_numeric(df['dislikes'], errors='coerce').fillna(0).astype(int)
df['comment_count']    = pd.to_numeric(df['comment_count'], errors='coerce').fillna(0).astype(int)
df['engagement_rate']  = ((df['likes'] + df['comment_count']) / df['views'].replace(0,1)).round(4)
df['like_ratio']       = (df['likes'] / (df['likes'] + df['dislikes']).replace(0,1)).round(4)
df['hashtag_count']    = df['tags'].fillna('').apply(lambda x: len(x.split('|')) if x != '[none]' else 0)
df['title_length']     = df['title'].fillna('').str.len()
df['duration_sec']     = np.random.choice([15,30,60,90,120,180], len(df))  # not in source

# Select and clean output columns
out_cols = [
    'video_id','title','channel_title','reel_category','category_name',
    'views','likes','dislikes','comment_count','engagement_rate','like_ratio',
    'hashtag_count','title_length','duration_sec','trending_date','thumbnail_link'
]
df_out = df[out_cols].drop_duplicates('video_id').reset_index(drop=True)
df_out.to_csv(f'{DEST}/labels.csv', index=True, index_label='sample_id')

print(f'✅ Reel Content labels.csv: {len(df_out):,} unique trending videos')
print(f'\nCategory distribution:')
print(df_out['reel_category'].value_counts())
print(f'\nEngagement stats:')
print(df_out[['views','likes','comment_count','engagement_rate']].describe().round(2))

✅ Reel Content labels.csv: 6,351 unique trending videos

Category distribution:
reel_category
dance        1619
other        1229
music         799
education     630
beauty        595
comedy        547
fitness       451
fashion       318
gaming        103
travel         60
Name: count, dtype: int64

Engagement stats:
             views       likes  comment_count  engagement_rate
count      6351.00     6351.00        6351.00          6351.00
mean     758209.56    34493.57        4501.71             0.05
std     1928993.10   116243.87       21460.22             0.04
min         549.00        0.00           0.00             0.00
25%       83511.00     1908.00         261.00             0.02
50%      270902.00     7987.00         921.00             0.04
75%      751266.50    25163.00        2845.00             0.06
max    48431654.00  3880071.00      733373.00             0.33


---
## 📊 STEP 10 — Final Summary Report

In [28]:
import os, pandas as pd

BASE = '/content/datasets'

check = [
    ('😊 Facial Expression — FER2013',  f'{BASE}/facial_expressions/fer2013'),
    ('😊 Facial Expression — RAF-DB',   f'{BASE}/facial_expressions/rafdb'),
    ('👁️  Eye Blink — MRL Eye',          f'{BASE}/eye_blink/mrl_eye'),
    ('👁️  Eye Blink — CEW',              f'{BASE}/eye_blink/cew'),
    ('📍 Facial Landmarks — 300W',       f'{BASE}/facial_landmarks/300w'),
    ('📍 Facial Landmarks — WFLW',       f'{BASE}/facial_landmarks/wflw'),
    ('📜 Scroll Behaviour',              f'{BASE}/scroll_behaviour'),
    ('🎬 Reel Content — YouTube',        f'{BASE}/reel_content'),
]

print('='*70)
print('                  📦  DATASET DOWNLOAD SUMMARY')
print('='*70)

total_images = 0
total_rows   = 0
total_mb     = 0

for name, path in check:
    if not os.path.exists(path):
        print(f'{name}')
        print(f'   ❌ Not downloaded')
        print()
        continue

    imgs = sum(1 for r,_,fs in os.walk(path)
               for f in fs if f.lower().endswith(('.jpg','.jpeg','.png','.bmp')))
    lbl  = os.path.join(path, 'labels.csv')
    rows = len(pd.read_csv(lbl)) if os.path.exists(lbl) else 0
    mb   = sum(os.path.getsize(os.path.join(r,f))
               for r,_,fs in os.walk(path) for f in fs) / 1e6
    lbl_status = '✅' if rows > 0 else '⚠️ '

    total_images += imgs; total_rows += rows; total_mb += mb

    print(f'{name}')
    print(f'   Images : {imgs:>8,}  |  Label rows : {rows:>8,}  |  Size : {mb:>7.1f} MB  {lbl_status}')
    print()

print('='*70)
print(f'  TOTAL  →  Images: {total_images:,}  |  Label rows: {total_rows:,}  |  Size: {total_mb:.1f} MB')
print('='*70)
print('\nAll labels.csv files saved inside each dataset folder.')
print('Run Step 11 below to save everything to Google Drive.')

                  📦  DATASET DOWNLOAD SUMMARY
😊 Facial Expression — FER2013
   Images :   35,887  |  Label rows :   35,887  |  Size :    59.2 MB  ✅

😊 Facial Expression — RAF-DB
   Images :        0  |  Label rows :        0  |  Size :     0.0 MB  ⚠️ 

👁️  Eye Blink — MRL Eye
   Images :   84,898  |  Label rows :   84,898  |  Size :   335.2 MB  ✅

👁️  Eye Blink — CEW
   Images :   48,000  |  Label rows :   48,000  |  Size :   182.4 MB  ✅

📍 Facial Landmarks — 300W
   Images :    2,204  |  Label rows :        0  |  Size :   546.0 MB  ⚠️ 

📍 Facial Landmarks — WFLW
   Images :        0  |  Label rows :        0  |  Size :     0.0 MB  ⚠️ 

📜 Scroll Behaviour
   Images :        0  |  Label rows :        0  |  Size :     0.0 MB  ⚠️ 

🎬 Reel Content — YouTube
   Images :        0  |  Label rows :    6,351  |  Size :   540.5 MB  ✅

  TOTAL  →  Images: 170,989  |  Label rows: 175,136  |  Size: 1663.2 MB

All labels.csv files saved inside each dataset folder.
Run Step 11 below to save everythin

---
## 💾 STEP 11 — (Optional) Save All Datasets to Google Drive

In [29]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

GDRIVE = '/content/drive/MyDrive/RealDatasets'
os.makedirs(GDRIVE, exist_ok=True)

datasets_to_save = [
    'facial_expressions',
    'eye_blink',
    'facial_landmarks',
    'scroll_behaviour',
    'reel_content'
]

print('📤 Copying to Google Drive...')
for ds in datasets_to_save:
    src = f'/content/datasets/{ds}'
    dst = f'{GDRIVE}/{ds}'
    if not os.path.exists(src):
        print(f'  ⚠️  {ds} not found — skipping')
        continue
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    mb = sum(os.path.getsize(os.path.join(r,f))
             for r,_,fs in os.walk(dst) for f in fs) / 1e6
    print(f'  ✅ {ds} → Google Drive  ({mb:.1f} MB)')

print(f'\n🎉 All datasets saved to: {GDRIVE}')

MessageError: Error: credential propagation was unsuccessful

In [30]:
import os, re, pandas as pd

DEST = '/content/datasets/facial_landmarks/300w'

# This dataset has faces.csv instead of .pts files — use that
faces_csv = f'{DEST}/faces.csv'

if os.path.exists(faces_csv):
    df = pd.read_csv(faces_csv)
    print('Columns:', df.columns.tolist())
    print(df.head(3))
    df.to_csv(f'{DEST}/labels.csv', index=False)
    print(f'✅ labels.csv saved: {len(df):,} rows')
else:
    # Build basic labels from image filenames
    rows = []
    img_dir = f'{DEST}/images'
    for img in sorted(os.listdir(img_dir)):
        if img.lower().endswith(('.jpg','.png')):
            rows.append({'filename': f'images/{img}', 'source': '300W'})
    df = pd.DataFrame(rows)
    df.to_csv(f'{DEST}/labels.csv', index=False)
    print(f'✅ labels.csv saved: {len(df):,} image entries')

Columns: ['image_name', 'width', 'height', 'x0', 'y0', 'x1', 'y1']
     image_name  width  height   x0   y0    x1   y1
0  00001722.jpg   1333    2000  490  320   687  664
1  00001044.jpg   2000    1333  791  119  1200  436
2  00001050.jpg    667    1000  304  155   407  331
✅ labels.csv saved: 3,350 rows


In [31]:
import os

DEST = '/content/datasets/facial_landmarks/wflw'

# Use a working Kaggle dataset with landmarks
os.system(f'kaggle datasets download -d "atulanandjha/lfpw" -p "{DEST}" --unzip')

# Check what downloaded
for root, dirs, files in os.walk(DEST):
    for f in files[:5]:
        print(os.path.join(root, f))

In [33]:
import pandas as pd, numpy as np, os

DEST = '/content/datasets/scroll_behaviour'
os.makedirs(DEST, exist_ok=True)
np.random.seed(42)

n = 10000
patterns = ['fast_scroll','slow_read','bounce','deep_read','skip_fold','skim']
devices  = ['mobile','tablet','desktop']

df = pd.DataFrame({
    'sample_id':        range(n),
    'scroll_pattern':   np.random.choice(patterns, n),
    'device':           np.random.choice(devices, n, p=[0.6,0.15,0.25]),
    'dwell_time_sec':   np.round(np.random.uniform(1, 300, n), 1),
    'scroll_depth_pct': np.random.randint(5, 101, n),
    'ctr':              np.round(np.random.uniform(0, 0.3, n), 4),
    'bounce':           np.random.randint(0, 2, n),
    'is_mobile':        0,
    'session_duration': np.round(np.random.uniform(5, 600, n), 1),
    'page_views':       np.random.randint(1, 20, n),
    'exit_rate':        np.round(np.random.uniform(0, 1, n), 4),
})
df['is_mobile'] = (df['device'] == 'mobile').astype(int)
df.to_csv(f'{DEST}/labels.csv', index=False)
print(f'✅ Scroll Behaviour labels.csv: {len(df):,} rows')
print(df['scroll_pattern'].value_counts())

✅ Scroll Behaviour labels.csv: 10,000 rows
scroll_pattern
slow_read      1692
skip_fold      1689
deep_read      1672
fast_scroll    1665
skim           1657
bounce         1625
Name: count, dtype: int64


In [34]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [37]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [38]:
!pip install opencv-python
!pip install tensorflow
!pip install kaggle
!pip install pandas
!pip install numpy

In [40]:
from google.colab import files
files.upload()

Saving adaptive_reel_detox.zip to adaptive_reel_detox.zip


{'adaptive_reel_detox.zip': b'PK\x03\x04\n\x00\x00\x00\x00\x00\xacym\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x14\x00\x1c\x00adaptive_reel_detox/UT\t\x00\x03\x94)\xb4i\x9c)\xb4iux\x0b\x00\x01\x04\x00\x00\x00\x00\x04\x00\x00\x00\x00PK\x03\x04\n\x00\x00\x00\x00\x00\xecxm\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x1a\x00\x1c\x00adaptive_reel_detox/utils/UT\t\x00\x03,(\xb4i\x9c)\xb4iux\x0b\x00\x01\x04\x00\x00\x00\x00\x04\x00\x00\x00\x00PK\x03\x04\x14\x00\x00\x00\x08\x00\xeaxm\\Rv\xfe\\\x12\r\x00\x00\xb7=\x00\x00\'\x00\x1c\x00adaptive_reel_detox/utils/db_manager.pyUT\t\x00\x03\'(\xb4i\'(\xb4iux\x0b\x00\x01\x04\x00\x00\x00\x00\x04\x00\x00\x00\x00\xd5\x1b\xdbn\xdb\xc8\xf5]_1\xf5b!rK\xd3\x92\xec\\V\x85\x128\x89\xb2k\xac/\xa9-o7\x08\x0cb,\x8e\xa4\xa9)RKRv\xdc \xc5\xf6\xa1o\x05\x8a\x16y(\x8a\x02\xdb\xfeE\xbf\'?\xd0~B\xcf\x19\xde\x86\xe4\x90\xa2\x1c%\xdb\x9d\xcd\xda"u\xce\x999\xf7sf\xc6[[[\xade\xc8\x9d`\xc7\xbe\xb4\xe6\xd4\xa5S\xe6\x9b\x8b\xdb\xd6\xa0<Zg\xbf>\xe4!#6\r\xe9%\r\x18I

In [44]:
from google.colab import files
files.upload()

Saving adaptive_reel_detox.zip to adaptive_reel_detox (1).zip


{'adaptive_reel_detox (1).zip': b'PK\x03\x04\n\x00\x00\x00\x00\x00\xacym\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x14\x00\x1c\x00adaptive_reel_detox/UT\t\x00\x03\x94)\xb4i\x9c)\xb4iux\x0b\x00\x01\x04\x00\x00\x00\x00\x04\x00\x00\x00\x00PK\x03\x04\n\x00\x00\x00\x00\x00\xecxm\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x1a\x00\x1c\x00adaptive_reel_detox/utils/UT\t\x00\x03,(\xb4i\x9c)\xb4iux\x0b\x00\x01\x04\x00\x00\x00\x00\x04\x00\x00\x00\x00PK\x03\x04\x14\x00\x00\x00\x08\x00\xeaxm\\Rv\xfe\\\x12\r\x00\x00\xb7=\x00\x00\'\x00\x1c\x00adaptive_reel_detox/utils/db_manager.pyUT\t\x00\x03\'(\xb4i\'(\xb4iux\x0b\x00\x01\x04\x00\x00\x00\x00\x04\x00\x00\x00\x00\xd5\x1b\xdbn\xdb\xc8\xf5]_1\xf5b!rK\xd3\x92\xec\\V\x85\x128\x89\xb2k\xac/\xa9-o7\x08\x0cb,\x8e\xa4\xa9)RKRv\xdc \xc5\xf6\xa1o\x05\x8a\x16y(\x8a\x02\xdb\xfeE\xbf\'?\xd0~B\xcf\x19\xde\x86\xe4\x90\xa2\x1c%\xdb\x9d\xcd\xda"u\xce\x999\xf7sf\xc6[[[\xade\xc8\x9d`\xc7\xbe\xb4\xe6\xd4\xa5S\xe6\x9b\x8b\xdb\xd6\xa0<Zg\xbf>\xe4!#6\r\xe9%\r\

In [45]:
!unzip adaptive_reel_detox.zip

Archive:  adaptive_reel_detox.zip
   creating: adaptive_reel_detox/
   creating: adaptive_reel_detox/utils/
  inflating: adaptive_reel_detox/utils/db_manager.py  
  inflating: adaptive_reel_detox/utils/addiction_scorer.py  
 extracting: adaptive_reel_detox/utils/__init__.py  
  inflating: adaptive_reel_detox/utils/detox_engine.py  
  inflating: adaptive_reel_detox/utils/face_processor.py  
   creating: adaptive_reel_detox/{static/
   creating: adaptive_reel_detox/{static/{css,js,images},templates,models,utils,data}/
   creating: adaptive_reel_detox/models/
  inflating: adaptive_reel_detox/models/train_blink_model.py  
  inflating: adaptive_reel_detox/models/train_emotion_model.py  
  inflating: adaptive_reel_detox/app.py  
  inflating: adaptive_reel_detox/requirements.txt  
  inflating: adaptive_reel_detox/README.md  
   creating: adaptive_reel_detox/templates/
  inflating: adaptive_reel_detox/templates/login.html  
  inflating: adaptive_reel_detox/templates/reels.html  
  inflating: a

In [46]:
%cd adaptive_reel_detox

/content/adaptive_reel_detox


In [47]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.15.0


In [48]:
!pip install flask opencv-python tensorflow numpy pandas pyngrok

In [49]:
!pip install pyngrok

In [51]:
!pip install pyngrok

In [52]:
from pyngrok import ngrok

ngrok.set_auth_token("3AtdUHeIBl0b1t52uEY2kvwInO1_2aCbwMCVL5RMMf21NnUb6")

In [55]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print("OPEN THIS WEBSITE:", public_url)

OPEN THIS WEBSITE: NgrokTunnel: "https://satisfiedly-nonbodily-meagan.ngrok-free.dev" -> "http://localhost:5000"


In [57]:
!pip install flask-cors

In [60]:
from pyngrok import ngrok
public_url = ngrok.connect(5000)
print(public_url)

NgrokTunnel: "https://satisfiedly-nonbodily-meagan.ngrok-free.dev" -> "http://localhost:5000"


In [62]:
!pip install flask flask-cors pyngrok

In [66]:
!python app.py

2026-03-13 15:43:37.138139: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-13 15:43:37.145234: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-13 15:43:37.167125: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773416617.210294   20136 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773416617.226060   20136 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773416617.260882   20136 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [67]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print(public_url)

NgrokTunnel: "https://satisfiedly-nonbodily-meagan.ngrok-free.dev" -> "http://localhost:5000"


In [68]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [69]:
!apt-get install git

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.15).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [70]:
!git config --global user.email "raksh9175@gmail.com"
!git config --global user.name "RakshithaMeleyyanavar"

In [75]:
!git clone https://github.com/RakshithaMeleyyanavar/Adaptive_reel_detox_system.git

Cloning into 'Adaptive_reel_detox_system'...
fatal: could not read Username for 'https://github.com': No such device or address


In [76]:
!cp *.ipynb Adaptive_reel_detox_system/

cp: cannot stat '*.ipynb': No such file or directory


In [77]:
%cd Adaptive_reel_detox_system

!git add .
!git commit -m "Upload Adaptive Reel Detox System project"
!git push

[Errno 2] No such file or directory: 'Adaptive_reel_detox_system'
/content/adaptive_reel_detox
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
